## AgentCore, Strands Agents 및 A2A 시작하기

[A2A protocol](https://a2a-protocol.org/dev/specification/)은 독립적이고 내부가 공개되지 않을 수 있는 AI agent system 간의 통신과 interoperability를 지원하도록 설계된 open standard입니다. 서로 다른 framework, language 또는 vendor를 사용하여 에이전트를 구축할 수 있는 ecosystem에서 A2A는 공통 언어와 상호 작용 모델을 제공합니다.

[Amazon AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)은 AI agent 또는 tool을 배포하고 실행하기 위한 안전한 serverless 전용 호스팅 환경을 제공합니다. 

AWS는 최근 AgentCore Runtime의 [A2A 지원](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html)을 발표했습니다.

이 workshop에서는 AgentCore Runtime을 사용하여 다음 아키텍처를 구축합니다.

<img src="images/architecture-getting-started.png" style="width: 80%;">

이 getting started Notebook에서는 두 에이전트를 구축합니다. 첫 번째 에이전트는 AWS Docs 전문가로, AWS Docs MCP를 query하여 AWS Documentation을 읽고 검색하며 권장 사항도 생성합니다. 두 번째 에이전트는 AWS Blog 전문가로, websearch를 사용하여 AWS 최신 blog와 news를 살펴봅니다.

이제 시작해 보겠습니다.

### 설정

dependency를 설치합니다.

In [ ]:
%pip install -q -r requirements.txt --no-cache-dir --force-reinstall

**새 version이 반영되도록 환경을 다시 시작하세요.**

In [ ]:
# import IPython

# IPython.Application.instance().kernel.do_shutdown(True)

`bedrock-agentcore-starter-toolkit` version이 0.1.21인지 확인합니다.

In [ ]:
!pip freeze | grep boto
!pip freeze | grep agentcore

In [ ]:
# library 가져오기
import json
import requests
from boto3.session import Session

# boto session 가져오기
boto_session = Session()

### 1 - 두 Agent의 코드 생성

`agents` 폴더가 없으면 생성합니다.

In [ ]:
![ ! -d "agents" ] && mkdir agents

#### 1.1 - AWS Docs Expert Agent

먼저 첫 번째 agent 코드를 로컬 파일에 작성합니다. 이 에이전트는 이후 AgentCore Runtime에 배포됩니다.

In [ ]:
%%writefile agents/strands_aws_docs.py
import os
import logging
import asyncio
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.multiagent.a2a import A2AServer
from strands.tools.mcp import MCPClient
from fastapi import FastAPI
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000

# lazy initialization을 사용하는 global MCP client
_mcp_client = None

async def get_mcp_client():
    """제한 시간을 적용해 MCP 클라이언트를 지연 초기화합니다."""
    global _mcp_client
    if _mcp_client is None:
        try:
            _mcp_client = MCPClient(
                lambda: stdio_client(
                    StdioServerParameters(
                        command="uvx", 
                        args=["awslabs.aws-documentation-mcp-server@latest"]
                    )
                )
            )
            # timeout과 함께 시작
            await asyncio.wait_for(_mcp_client.start(), timeout=10.0)
            logger.info("MCP client initialized")
        except asyncio.TimeoutError:
            logger.error("MCP client startup timed out")
            _mcp_client = None
        except Exception as e:
            logger.error(f"MCP client failed: {e}")
            _mcp_client = None
    return _mcp_client

system_prompt = """You are an AWS Documentation Assistant powered by the AWS Documentation MCP server. Your role is to help users find accurate, up-to-date information from AWS documentation.

CRITICAL: Keep responses SHORT and FOCUSED.

Guidelines:
- Provide concise, actionable answers (max 3 sentences)
- Use bullet points for lists
- Skip verbose explanations
- If MCP is unavailable, provide basic AWS knowledge
- Timeout operations after 8 seconds
- Prioritize speed over completeness

You have access to AWS documentation search tools when available."""

# 먼저 최소 tool로 agent 초기화
agent = Agent(
    system_prompt=system_prompt, 
    tools=[],  # tool 없이 시작하고 동적으로 추가
    name="AWS Docs Agent",
    description="An agent to query AWS Docs using AWS MCP.",
)

# MCP가 준비되면 tool을 동적으로 추가
async def setup_agent_tools():
    """MCP 클라이언트가 준비되면 에이전트 도구를 설정합니다."""
    try:
        mcp_client = await get_mcp_client()
        if mcp_client:
            tools = await asyncio.wait_for(
                mcp_client.list_tools_async(), 
                timeout=5.0
            )
            agent.tools = [tools] if tools else []
            logger.info("Agent tools configured")
    except Exception as e:
        logger.warning(f"Could not setup MCP tools: {e}")

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

@app.on_event("startup")
async def startup_event():
    """시작 시 MCP 클라이언트를 초기화합니다."""
    await setup_agent_tools()

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

#### **선택 사항** - 로컬 테스트

이 코드를 로컬에서 테스트하려면 bash/terminal 창을 열고 다음 snippet을 실행합니다.

```bash
python agents/strands_aws_docs.py
```

server가 로컬에서 시작됩니다. 그런 다음 다른 terminal/bash에서 다음 명령을 실행하여 테스트합니다.

```bash
curl -X POST http://0.0.0.0:9000 \-H "Content-Type: application/json" \-d '{  "jsonrpc": "2.0",  "id": "req-001",  "method": "message/send",  "params": {  "message": {  "role": "user",  "parts": [  {  "kind": "text",  "text": "What's AWS Lambda?"  }  ],  "messageId": "d0673ab9-796d-4270-9435-451912020cd1"  }  } }' | jq .
```

MCP를 query한 다음 AWS Lambda를 설명하는 답변을 반환합니다.

다음 명령을 사용하여 agent card 정보 가져오기도 테스트할 수 있습니다.

```bash
curl http://localhost:9000/.well-known/agent-card.json | jq .
```

#### 1.2 - AWS Blogs Expert Agent

두 번째 agent 코드를 로컬 파일에 작성합니다.

In [ ]:
%%writefile agents/strands_aws_blogs_news.py
import logging
import os
import asyncio
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
import uvicorn
from fastapi import FastAPI

from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')

@tool
async def fast_internet_search(keywords: str, max_results: int = 3) -> str:
    """Fast web search with timeouts.
    Args:
        keywords (str): Search query keywords
        max_results (int): Max results (default 3 for speed)
    Returns:
        Search results
    """
    try:
        # 더 나은 결과를 위해 AWS 관련 용어 추가
        aws_keywords = f"site:aws.amazon.com {keywords} AWS"
        
        # 검색에 asyncio timeout 사용
        async def search_with_timeout():
            return DDGS().text(
                aws_keywords, 
                region="us-en", 
                max_results=max_results
            )
        
        results = await asyncio.wait_for(search_with_timeout(), timeout=8.0)
        
        if results:
            # 결과를 간결한 형식으로 지정
            formatted = []
            for i, result in enumerate(results[:max_results], 1):
                formatted.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('href', '')}")
            
            return "\n".join(formatted)
        else:
            return "No AWS results found."
            
    except asyncio.TimeoutError:
        logger.warning(f"Search timeout for: {keywords}")
        return "Search timed out. Try a more specific query."
    except RatelimitException:
        logger.warning("Rate limit hit")
        return "Rate limit reached. Please try again in a moment."
    except (DDGSException, Exception) as e:
        logger.error(f"Search error: {e}")
        return f"Search unavailable: {str(e)[:50]}"

system_prompt = """You are an AWS Blog Expert.

CRITICAL: Keep responses SHORT and RECENT.

Guidelines:
- Provide max 3 recent results
- Focus on official AWS blog posts only
- Use concise summaries (1-2 sentences per result)
- Include direct links when available
- Timeout searches after 8 seconds
- If search fails, acknowledge limitation

Search Strategy:
- Always include "AWS" in searches
- Focus on aws.amazon.com/blogs/ content
- Prioritize recent announcements"""

agent = Agent(
    system_prompt=system_prompt, 
    tools=[fast_internet_search],
    name="AWS Blog/News Agent",
    description="An agent to search on Web latest AWS Blogs and News.",
)

host, port = "0.0.0.0", 9000

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

에이전트에 필요한 dependency가 포함된 requirements.txt 파일을 작성합니다.

In [ ]:
%%writefile agents/requirements.txt
boto3==1.40.50
bedrock-agentcore==0.1.7
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs

### 2 - AgentCore Runtime에 배포

이 solution을 AgentCore Runtime에 배포합니다.

#### 2.1 - Cognito User Pool 설정

에이전트를 배포하기 전에 에이전트에 액세스하는 사용자를 검증할 수 있도록 Cognito User Pool 또는 Okta, Microsoft Entra ID 같은 다른 Identity provider를 설정해야 합니다.

workshop의 몇 가지 단계를 간소화하는 method가 포함된 helper class를 가져옵니다. 이 helper class는 Cognito User Pool 생성을 담당하는 method를 가져옵니다.

In [ ]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()  # 이 output cell에서 bearer token을 가져옴
print("Cognito setup completed ✓")

#### 2.2 - Agent용 IAM Role 생성

##### 2.2.1 AWS Docs Agent Execution Role

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_DOCS_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)

##### 2.2.2 AWS Blogs Agent Execution Role

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_BLOG_ROLE_NAME

execution_role_arn_blogs = create_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)

##### AgentCore Runtime 배포 구성 생성

다음 섹션에서는 [starter toolkit](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-starter-toolkit.html)을 활용합니다. starter toolkit은 AI agent를 AgentCore Runtime에 배포하는 데 사용할 수 있는 Command Line Interface(CLI) toolkit입니다.

이제 AgentCore Runtime 내부에서 A2A protocol을 지원하는 에이전트를 생성합니다.

##### 2.2.3 - 첫 번째 Agent 구성 및 배포(AWS Docs Agent)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_mcp_agent = Runtime()
aws_docs_agent_name = "aws_docs_assistant"

region = boto_session.region_name

# 배포 구성
response_aws_docs_agent = agentcore_runtime_mcp_agent.configure(
    entrypoint="agents/strands_aws_docs.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_docs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("Configuration completed:", response_aws_docs_agent)

첫 번째 에이전트를 AgentCore Runtime에 시작합니다.

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

docs_agent_arn = launch_result_mcp.agent_arn

**배포 상태 확인**

배포가 완료되었는지 확인합니다.

In [ ]:
status_response = agentcore_runtime_mcp_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

##### 2.2.4 - 두 번째 Agent 구성 및 배포(AWS Blogs and News Agent)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_blogs = Runtime()
aws_blogs_agent_name = "aws_blog_assistant"

# 배포 구성
response_aws_blogs_agent = agentcore_runtime_blogs.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn_blogs,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_blogs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("Configuration completed:", response_aws_blogs_agent)

두 번째 에이전트를 AgentCore Runtime에 시작합니다.

In [ ]:
launch_result_blog = agentcore_runtime_blogs.launch()
print("Launch completed:", launch_result_blog.agent_arn)

blog_agent_arn = launch_result_blog.agent_arn

**배포 상태 확인**

두 번째 에이전트의 배포가 완료되었는지 확인합니다.

In [ ]:
status_response = agentcore_runtime_blogs.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

##### 2.2.5 - Output 내보내기 및 저장

다음 Notebook에서 사용할 variable을 내보냅니다.

In [ ]:
MCP_AGENT_ID = launch_result_mcp.agent_id
MCP_AGENT_ARN = launch_result_mcp.agent_arn
MCP_AGENT_NAME = aws_docs_agent_name

BLOG_AGENT_ID = launch_result_blog.agent_id
BLOG_AGENT_ARN = launch_result_blog.agent_arn
BLOG_AGENT_NAME = aws_blogs_agent_name

COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store MCP_AGENT_ID
%store MCP_AGENT_ARN
%store MCP_AGENT_NAME
%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

orchestrator에서 사용할 수 있도록 agent ARN을 SSM에 저장합니다.

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, MCP_AGENT_ARN)

put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

### 3 - A2A Agent 호출

먼저 auth token을 갱신합니다.

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

#### 3.1 Agent Card 가져오기

첫 번째 에이전트(AWS Docs MCP Expert)에서 Agent Card 정보를 가져옵니다.

In [ ]:
import logging
from uuid import uuid4
from urllib.parse import quote

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)


def fetch_agent_card(agent_arn):
    # agent ARN을 URL encoding
    escaped_agent_arn = quote(agent_arn, safe="")

    # URL 구성
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    logger.info(url)
    # 고유한 session ID 생성
    session_id = str(uuid4())
    logger.info(f"Generated session ID: {session_id}")

    # header 설정
    headers = {
        "Accept": "*/*",
        "Authorization": f"Bearer {bearer_token}",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
        "X-Amzn-Trace-Id": f"aws_docs_assistant_{session_id}",
    }

    try:
        # request 실행
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # JSON을 parse하여 보기 좋게 출력
        agent_card = response.json()
        logger.info(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching agent card: {e}")
        return None

In [ ]:
fetch_agent_card(docs_agent_arn)

두 번째 에이전트(AWS Blogs and News expert)의 agent card를 확인합니다. 

In [ ]:
fetch_agent_card(blog_agent_arn)

#### 3.2 - Agent 테스트

A2A를 사용하여 첫 번째 에이전트를 호출합니다.

In [ ]:
import logging

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # request timeout을 5 minutes로 설정


def format_agent_response(response):
    """에이전트 응답을 추출해 읽기 쉬운 형식으로 변환합니다."""
    # artifact에서 주요 response text 가져오기
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text

    # fallback: history의 모든 agent message 연결
    agent_messages = [msg.parts[0].root.text for msg in response.history if msg.role.value == "agent" and msg.parts]
    return "".join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )


async def send_sync_message(agent_arn, message: str):
    # environment variable에서 runtime URL 가져오기
    escaped_agent_arn = quote(agent_arn, safe="")

    # URL 구성
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"

    # 고유한 session ID 생성
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # AgentCore용 authentication header 추가
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # runtime URL에서 agent card 가져오기
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card에 올바른 URL이 포함됨(이 경우 runtime_url과 동일)
        # 수동 override 불필요 - path 기반 mounting pattern

        # factory를 사용하여 client 생성
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # sync response에 non-streaming mode 사용
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # message 생성 및 전송
        msg = create_message(text=message)

        # streaming=False이면 정확히 하나의 결과가 생성됨
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) 튜플
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # 다른 response type용 fallback
                logger.info(f"Response: {str(event)}")
                return event

In [ ]:
result = await send_sync_message(docs_agent_arn, "what is DynamoDB")
formatted_output = format_agent_response(result)
print(formatted_output)

두 번째 에이전트를 테스트합니다.

In [ ]:
result = await send_sync_message(blog_agent_arn, "Give me the latest published blog for Bedrock AgentCore?")
formatted_output = format_agent_response(result)
print(formatted_output)

다음은 에이전트가 수행한 단계를 보여주는 더 상세한 output입니다.

에이전트에 묻는 질문을 자유롭게 변경하고 단계별 결과를 확인해 보세요.

In [ ]:
def format_agent_trace(response):
    """에이전트 응답을 읽기 쉬운 호출 추적 형식으로 변환합니다."""
    print("=" * 60)
    print("🔍 AGENT EXECUTION TRACE")
    print("=" * 60)

    # Context 정보
    print(f"📋 Context ID: {response.context_id}")
    print(f"🆔 Task ID: {response.id}")
    print(f"📊 Status: {response.status.state.value}")
    print(f"⏰ Completed: {response.status.timestamp}")
    print()

    # history 추적
    print("🔄 EXECUTION FLOW:")
    print("-" * 40)

    for i, msg in enumerate(response.history, 1):
        role_icon = "👤" if msg.role.value == "user" else "🤖"
        text = msg.parts[0].root.text if msg.parts else "[No content]"

        # trace view에서 긴 message 자르기
        if len(text) > 80:
            text = text[:77] + "..."

        print(f"{i:2d}. {role_icon} {msg.role.value.upper()}: {text}")

    print()
    print("✅ FINAL RESULT:")
    print("-" * 40)

    # 최종 artifact
    if response.artifacts:
        final_text = response.artifacts[0].parts[0].root.text
        print(final_text[:200] + "..." if len(final_text) > 200 else final_text)

    print("=" * 60)

In [ ]:
format_agent_trace(result)

축하합니다. A2A protocol을 사용하여 첫 번째 에이전트를 Amazon AgentCore Runtime에 배포했습니다.

이제 다음 lab으로 이동합니다.